# 🚀 HalfLife: Real-World Temporal RAG Demo

**Does your RAG return outdated answers? This notebook proves it — and fixes it.**

We fetch **real research papers from Arxiv spanning 2019–2024**, ingest them into a vector store,
and show side-by-side: what standard vector search returns vs what HalfLife returns after temporal reranking.

No config needed. Just run all cells.

---
[![PyPI](https://img.shields.io/pypi/v/halflife-rag)](https://pypi.org/project/halflife-rag/)  
GitHub: [amaydixit11/halflife](https://github.com/amaydixit11/halflife)


In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q halflife-rag sentence-transformers qdrant-client redis rich
!apt-get install -q -y redis-server && service redis-server start
print('✅ Done.')

## 📡 Cell 2: Fetch a temporally-spread corpus

**Why year-sliced fetching?** Arxiv's API sorts by `submittedDate desc` by default,
so a naive fetch returns only papers from the last few weeks — no temporal spread, no demo.

Instead we fetch **~15–25 papers per year** from 2019–2024 using Arxiv's `submittedDate`
filter, restricted to `cs.CL` (Computation and Language). This gives us real papers
from the BERT era (2019), the adapter era (2020), the LoRA era (2021–22),
and the QLoRA/DoRA era (2023–24) — exactly the temporal gap HalfLife is built to exploit.


In [ ]:
import requests, xml.etree.ElementTree as ET, time
from datetime import datetime, timezone
from dataclasses import dataclass
from typing import List
from collections import Counter

@dataclass
class Paper:
    title: str
    abstract: str
    published: datetime
    arxiv_id: str

def fetch_arxiv_slice(query: str, year: int, n: int = 20) -> List[Paper]:
    """Fetch papers from a single year using Arxiv date filter."""
    date_filt  = f"submittedDate:[{year}0101 TO {year}1231]"
    full_query = f"({query}) AND {date_filt}"
    params = {"search_query": full_query, "start": 0,
               "max_results": n, "sortBy": "relevance"}
    r = requests.get("http://export.arxiv.org/api/query", params=params, timeout=30)
    r.raise_for_status()
    ns   = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(r.content)
    out  = []
    for e in root.findall("atom:entry", ns):
        title = e.find("atom:title",   ns).text.strip().replace("\n", " ")
        abst  = e.find("atom:summary", ns).text.strip().replace("\n", " ")
        pub   = datetime.strptime(e.find("atom:published", ns).text,
                                   "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
        aid   = e.find("atom:id", ns).text.strip()
        out.append(Paper(title=title, abstract=abst, published=pub, arxiv_id=aid))
    return out

# Narrow cs.CL query — avoids cross-domain contamination (no physics papers)
QUERY = 'ti:"fine-tuning" AND (ti:"language model" OR ti:"transformer") AND cat:cs.CL'

# (year, papers_to_fetch) — more recent years have more relevant work so we fetch more
SLICES = [(2019,15),(2020,15),(2021,20),(2022,20),(2023,25),(2024,25)]

all_papers, seen = [], set()
print("📡 Fetching year-sliced corpus from Arxiv cs.CL...\n")
for yr, n in SLICES:
    papers = fetch_arxiv_slice(QUERY, yr, n)
    new = [p for p in papers if p.arxiv_id not in seen]
    seen.update(p.arxiv_id for p in new)
    all_papers.extend(new)
    sample = f"e.g. '{new[0].title[:55]}...'" if new else "(none returned)"
    print(f"  {yr}: {len(new):2d} papers  {sample}")
    time.sleep(3)

print(f"\n✅ Total: {len(all_papers)} papers")
years = Counter(p.published.year for p in all_papers)
print("\n📅 Year distribution (this spread is what makes the demo work):")
for yr in sorted(years):
    print(f"  {yr}: {'█' * years[yr]} ({years[yr]})")

## 🔧 Cell 3: Ingest — vectors into Qdrant, metadata into Redis

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest
from sentence_transformers import SentenceTransformer
import uuid, redis, json, math

qdrant = QdrantClient(":memory:")
COLL   = "arxiv_finetune"
qdrant.create_collection(collection_name=COLL,
    vectors_config=rest.VectorParams(size=384, distance=rest.Distance.COSINE))

rc = redis.Redis(host="localhost", port=6379, decode_responses=True)
NOW = datetime.now(timezone.utc)

print("🔄 Loading embedding model...")
emb = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Model loaded. Ingesting {len(all_papers)} papers...")

# Decay rate by age:
#   old papers (>3yr): slow decay — foundational work stays relevant for longer
#   mid papers (1-3yr): standard
#   recent (<1yr): slightly faster — not yet proven longevity
def pick_lambda(pub):
    age = (NOW - pub).days
    return 5e-8 if age > 1095 else (1e-7 if age > 365 else 3e-7)

pts = []
for i, p in enumerate(all_papers):
    text = f"{p.title}. {p.abstract[:500]}"
    cid  = str(uuid.uuid4())
    pts.append(rest.PointStruct(
        id=cid, vector=emb.encode(text).tolist(),
        payload={"title": p.title, "text": text,
                 "timestamp": p.published.isoformat(),
                 "year": p.published.year, "arxiv_id": p.arxiv_id}
    ))
    rc.set(f"chunk:{cid}", json.dumps({
        "chunk_id": cid, "decay_type": "exponential",
        "decay_params": {"lambda": pick_lambda(p.published)},
        "trust_score": 0.85
    }))
    if len(pts) == 32 or i == len(all_papers)-1:
        qdrant.upsert(collection_name=COLL, points=pts); pts = []
        if (i+1) % 50 == 0: print(f"  ...{i+1}/{len(all_papers)}")

print(f"\n✅ Ingested. Vector store + Redis ready.")

## 🔥 Cell 4: The Side-by-Side Demo

Each query runs twice: once with pure cosine similarity (Baseline),
once with HalfLife temporal fusion. The **year column tells the story**.

Color guide: 🟢 = correct era for the intent | 🟡 = acceptable | 🔴 = wrong era


In [ ]:
import statistics
from rich.console import Console
from rich.table import Table
from rich import box
console = Console()

# ── Reranker ──────────────────────────────────────────────────────────────────
FRESH_KW = {"latest","current","best","modern","today","sota",
             "state-of-the-art","now","recent","2024","2025"}
HIST_KW  = {"original","first","early","originally","how was",
             "history","was done","bert","adapter"}

def classify(q):
    ql = q.lower()
    if any(k in ql for k in FRESH_KW): return "fresh",    {"vector":0.3,"temporal":0.6,"trust":0.1}
    if any(k in ql for k in HIST_KW):  return "historical",{"vector":0.4,"temporal":0.5,"trust":0.1}
    return "static", {"vector":0.8,"temporal":0.1,"trust":0.1}

def tdecay(ts_iso, lam, inv):
    try:
        ts = datetime.fromisoformat(ts_iso)
        if ts.tzinfo is None: ts = ts.replace(tzinfo=timezone.utc)
        raw = math.exp(-lam * max(0,(NOW-ts).total_seconds()))
        return (1-raw) if inv else raw
    except: return 0.5

def halflife_rerank(query, hits, top_k=5):
    intent, w = classify(query)
    inv = (intent == "historical")
    scored = []
    for r in hits:
        cid  = str(r.id)
        meta = json.loads(rc.get(f"chunk:{cid}") or "{}")
        lam  = meta.get("decay_params",{}).get("lambda",1e-7)
        tr   = meta.get("trust_score",0.5)
        ts   = r.payload.get("timestamp","")
        scored.append({"id":cid,"title":r.payload.get("title",""),
                        "year":r.payload.get("year",0),
                        "v":r.score,"t":tdecay(ts,lam,inv),"tr":tr})
    vs = [s["v"] for s in scored]; ts2 = [s["t"] for s in scored]
    def nm(x,xs): lo,hi=min(xs),max(xs); return (x-lo)/(hi-lo) if hi>lo else 0.5
    for s in scored:
        s["final"] = w["vector"]*nm(s["v"],vs)+w["temporal"]*nm(s["t"],ts2)+w["trust"]*s["tr"]
    scored.sort(key=lambda x:x["final"],reverse=True)
    return scored[:top_k], intent

def ycolor(yr, intent):
    if intent=="fresh":      return "green" if yr>=2023 else ("yellow" if yr>=2021 else "red")
    if intent=="historical": return "green" if yr<=2020 else ("yellow" if yr<=2022 else "red")
    return "white"

# ── Queries ───────────────────────────────────────────────────────────────────
QUERIES = [
    # FRESH — right answer is 2023/2024 papers
    "Best current approach to parameter-efficient fine-tuning today?",
    "State-of-the-art method for efficient fine-tuning of large language models?",
    "Most recent advances in fine-tuning pretrained transformers?",
    # HISTORICAL — right answer is 2019/2020 papers
    "How was fine-tuning of BERT originally done in early NLP research?",
    "What were the original adapter-based approaches for fine-tuning?",
    # STATIC — should not regress vs baseline
    "Explain how fine-tuning differs from pretraining in language models",
]

b_ages = {"fresh":[],"historical":[],"static":[]}
h_ages = {"fresh":[],"historical":[],"static":[]}

for query in QUERIES:
    q_vec = emb.encode(query).tolist()
    hits  = qdrant.query_points(collection_name=COLL, query=q_vec,
                                 limit=20, with_payload=True).points
    hl, intent = halflife_rerank(query, hits, top_k=5)

    t = Table(title=f'"{query[:72]}"  [{intent.upper()}]', box=box.SIMPLE_HEAVY)
    t.add_column("#",      width=3)
    t.add_column("Method", width=12)
    t.add_column("Year",   width=6)
    t.add_column("Age",    width=6)
    t.add_column("Score",  width=7)
    t.add_column("Title",  width=60)

    for i, r in enumerate(hits[:3]):
        yr = r.payload.get("year",0); c = ycolor(yr, intent)
        t.add_row(str(i+1),"[dim]Baseline[/dim]",
                  f"[{c}]{yr}[/]",f"[{c}]{NOW.year-yr}yr[/]",
                  f"{r.score:.3f}",r.payload.get("title","")[:60])
        b_ages[intent].append(NOW.year-yr)
    t.add_section()
    for i, r in enumerate(hl[:3]):
        yr = r["year"]; c = ycolor(yr, intent)
        t.add_row(str(i+1),"[bold green]HalfLife[/bold green]",
                  f"[{c}]{yr}[/]",f"[{c}]{NOW.year-yr}yr[/]",
                  f"{r['final']:.3f}",r["title"][:60])
        h_ages[intent].append(NOW.year-yr)

    console.print(t); console.print()

## 📊 Cell 5: Aggregate Metrics

In [ ]:
def sm(xs): return statistics.mean(xs) if xs else 0.0

s = Table(title="📊 HalfLife vs Baseline — Real Arxiv Corpus (cs.CL 2019–2024)",
          box=box.HEAVY_EDGE)
s.add_column("Intent",   style="cyan", width=12)
s.add_column("Measure",  width=34)
s.add_column("Baseline", justify="right", width=12)
s.add_column("HalfLife", justify="right", style="bold green", width=12)
s.add_column("Δ",        justify="right", width=10)
s.add_column("Verdict",  width=28)

for intent in ["fresh","historical","static"]:
    b, h = sm(b_ages[intent]), sm(h_ages[intent])
    d = b - h  # positive = HL is MORE recent
    if intent == "fresh":
        label   = "Avg result age ↓ (lower = more recent)"
        verdict = (f"✅ {d:.1f}yr fresher" if d>0.4 else
                   "⚠️  Marginal" if d>0 else "❌ No improvement")
    elif intent == "historical":
        label   = "Avg result age ↑ (higher = more historical)"
        verdict = (f"✅ {-d:.1f}yr older" if d<-0.4 else
                   "⚠️  Marginal" if d<0 else "❌ No improvement")
    else:
        label   = "Avg result age ~ (should be stable)"
        verdict = "✅ Stable" if abs(d)<0.5 else "⚠️  Some drift"
    s.add_row(intent.upper(), label,
              f"{b:.1f}yr", f"{h:.1f}yr", f"{-d:+.1f}yr", verdict)

console.print(s)
console.print()
console.print("[bold cyan]Reading the Δ column:[/bold cyan]")
console.print("  FRESH:      negative Δ = HalfLife is more recent  ✅")
console.print("  HISTORICAL: positive Δ = HalfLife is older (correct) ✅")
console.print("  STATIC:     near-zero Δ = no regression ✅")

## 🎯 Cell 6: The Authority Trap

The hardest test. We find the oldest and newest papers in our corpus and show
which era a fresh query surfaces under baseline vs HalfLife.


In [ ]:
sp = sorted(all_papers, key=lambda p: p.published)

console.print("[bold yellow]⚠️  Corpus Era Anchors[/bold yellow]\n")
console.print("[dim]Oldest papers (the 'authority trap' — high cosine sim, wrong era):[/dim]")
for p in sp[:4]: console.print(f"  [red]{p.published.year}[/red]  {p.title[:72]}")
console.print("\n[dim]Newest papers (correct answer for a FRESH query):[/dim]")
for p in sp[-4:]: console.print(f"  [green]{p.published.year}[/green]  {p.title[:72]}")

trap_q = "What is the best current method for fine-tuning pre-trained language models?"
q_vec  = emb.encode(trap_q).tolist()
hits   = qdrant.query_points(collection_name=COLL, query=q_vec, limit=20, with_payload=True).points
hl, _  = halflife_rerank(trap_q, hits, top_k=5)

console.print(f"\n[bold]Trap query:[/bold] '{trap_q}'\n")
tt = Table(box=box.SIMPLE)
tt.add_column("Rank",   width=6)
tt.add_column("Method", width=12)
tt.add_column("Year",   width=6)
tt.add_column("Age",    width=6)
tt.add_column("CosSim", width=7)
tt.add_column("Final",  width=7)
tt.add_column("Title",  width=62)

for i,r in enumerate(hits[:5]):
    yr=r.payload.get("year",0); c=ycolor(yr,"fresh")
    tt.add_row(f"#{i+1}","[dim]Baseline[/dim]",
               f"[{c}]{yr}[/]",f"{NOW.year-yr}yr",f"{r.score:.3f}","—",r.payload.get("title","")[:62])
tt.add_section()
for i,r in enumerate(hl[:5]):
    yr=r["year"]; c=ycolor(yr,"fresh")
    tt.add_row(f"#{i+1}","[bold green]HalfLife[/bold green]",
               f"[{c}]{yr}[/]",f"{NOW.year-yr}yr",f"{r['v']:.3f}",f"{r['final']:.3f}",r["title"][:62])

console.print(tt)

b_yr = hits[0].payload.get("year",0)
h_yr = hl[0]["year"]
gap  = h_yr - b_yr
if gap > 0:
    console.print(f"\n[bold green]✅ HalfLife promoted a result {gap} year(s) newer than baseline's #1.[/bold green]")
elif gap == 0:
    console.print(f"\n[yellow]Both agree on the top year ({b_yr}). Check the full ranking — reordering within the same year is still meaningful.[/yellow]")
else:
    console.print(f"\n[dim]Baseline was already recent for this query. Look at the HISTORICAL queries above for the inversion effect.[/dim]")

## 🦜 Cell 7: Drop-in Integration

In [ ]:
print("""
── LangChain ───────────────────────────────────────────────────────────────
  from langchain.retrievers import ContextualCompressionRetriever
  from halflife.integrations.langchain import HalfLifeReranker

  retriever = ContextualCompressionRetriever(
      base_compressor=HalfLifeReranker(top_k=5),
      base_retriever=your_existing_retriever
  )
  docs = retriever.get_relevant_documents("Latest fine-tuning method?")

── LlamaIndex ──────────────────────────────────────────────────────────────
  from halflife.integrations.llamaindex import HalfLifePostprocessor

  query_engine = index.as_query_engine(
      similarity_top_k=20,
      node_postprocessors=[HalfLifePostprocessor(top_n=5)]
  )

── Bare SDK ────────────────────────────────────────────────────────────────
  from halflife import HalfLife
  hl = HalfLife()
  reranked = hl.rerank(query=query, chunks=qdrant_results, top_k=5)

Install: pip install halflife-rag
Repo:    https://github.com/amaydixit11/halflife
""")